In [92]:
#mount google drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [94]:
import tensorflow as tf
import numpy as np
import os
import math

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [95]:
BASE_PATH = "/content/drive/MyDrive/Translation Language Translator "

TOKENIZED_PATH = f"{BASE_PATH}/tokenized"

print(TOKENIZED_PATH)

/content/drive/MyDrive/Translation Language Translator /tokenized


In [96]:
#load tokenizer information
# the transformer needs to know the vobabulary sizes.

import glob
chunk_files = sorted(
    glob.glob(f"{TOKENIZED_PATH}/train_chunk_*.npz")
)

print("Number of chunks:", len(chunk_files))

print("\nFirst 5 files:")
for file in chunk_files[:5]:
    print(file)

Number of chunks: 14

First 5 files:
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_000.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_001.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_002.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_003.npz
/content/drive/MyDrive/Translation Language Translator /tokenized/train_chunk_004.npz


In [97]:
#inspect first chunk
sample_chunk = np.load(chunk_files[0])

print("Arrays inside chunk:")
print(sample_chunk.files)

print("\nEncoder input shape:")
print(sample_chunk["encoder_inputs"].shape)

print("\nDecoder input shape:")
print(sample_chunk["decoder_inputs"].shape)

print("\nDecoder target shape:")
print(sample_chunk["decoder_targets"].shape)

Arrays inside chunk:
['encoder_inputs', 'decoder_inputs', 'decoder_targets']

Encoder input shape:
(100000, 56)

Decoder input shape:
(100000, 64)

Decoder target shape:
(100000, 64)


In [98]:
MAX_ENGLISH_LEN = sample_chunk["encoder_inputs"].shape[1]
MAX_HINDI_LEN = sample_chunk["decoder_inputs"].shape[1]

print("MAX_ENGLISH_LEN:", MAX_ENGLISH_LEN)
print("MAX_HINDI_LEN:", MAX_HINDI_LEN)

MAX_ENGLISH_LEN: 56
MAX_HINDI_LEN: 64


In [99]:
#transformer configuration
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
D_FF = 512
DROPOUT_RATE=0.1

In [100]:
assert D_MODEL % NUM_HEADS == 0

HEAD_DIM = D_MODEL // NUM_HEADS

print("D_MODEL:", D_MODEL)
print("NUM_HEADS:", NUM_HEADS)
print("HEAD_DIM:", HEAD_DIM)
print("D_FF:", D_FF)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)

D_MODEL: 128
NUM_HEADS: 4
HEAD_DIM: 32
D_FF: 512
Encoder layers: 2
Decoder layers: 2


In [101]:
#vocabulary size
ENGLISH_VOCAB_SIZE = 16000
HINDI_VOCAB_SIZE = 16000

print("English vocabulary:", ENGLISH_VOCAB_SIZE)
print("Hindi vocabulary:", HINDI_VOCAB_SIZE)

English vocabulary: 16000
Hindi vocabulary: 16000


In [102]:
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [103]:
#positional encoding
def positional_encoding(max_length, d_model):

    positions = np.arange(max_length)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angle_rates = 1 / np.power(
        10000,
        (2 * (dimensions // 2)) / np.float32(d_model)
    )

    angle_radians = positions * angle_rates

    encoding = np.zeros(
        (max_length, d_model)
    )

    encoding[:, 0::2] = np.sin(
        angle_radians[:, 0::2]
    )

    encoding[:, 1::2] = np.cos(
        angle_radians[:, 1::2]
    )

    return tf.cast(
        encoding,
        dtype=tf.float32
    )

In [104]:
#generate positional encoding
encoder_pos_encoding = positional_encoding(
    MAX_ENGLISH_LEN,
    D_MODEL
)

decoder_pos_encoding = positional_encoding(
    MAX_HINDI_LEN,
    D_MODEL
)

print(
    "Encoder positional encoding:",
    encoder_pos_encoding.shape
)

print(
    "Decoder positional encoding:",
    decoder_pos_encoding.shape
)

Encoder positional encoding: (56, 128)
Decoder positional encoding: (64, 128)


In [105]:
#token embedding
#token IDs are converted into vectors
class TokenEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, d_model):
        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=d_model
        )

        self.d_model = d_model

    def call(self, x):

        x = self.embedding(x)

        # Scale embedding according to Transformer paper
        x *= tf.math.sqrt(
            tf.cast(self.d_model, tf.float32)
        )

        return x

In [106]:
#scsled DOT-Product attention
def scaled_dot_product_attention(
    query,
    key,
    value,
    mask=None
):

    # Q × Kᵀ
    scores = tf.matmul(
        query,
        key,
        transpose_b=True
    )

    # √d_k
    depth = tf.cast(
        tf.shape(key)[-1],
        tf.float32
    )

    # Scale
    scores = scores / tf.math.sqrt(depth)

    # Apply mask
    if mask is not None:

        scores += (
            mask * -1e9
        )

    # Softmax
    attention_weights = tf.nn.softmax(
        scores,
        axis=-1
    )

    # Weighted sum of V
    output = tf.matmul(
        attention_weights,
        value
    )

    return output, attention_weights

In [107]:
#multi head attention  each head operates on 128 / 4 = 32D
class MultiHeadAttention(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        d_model,
        num_heads
    ):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.depth = d_model // num_heads

        # Q, K, V projections
        self.wq = tf.keras.layers.Dense(
            d_model
        )

        self.wk = tf.keras.layers.Dense(
            d_model
        )

        self.wv = tf.keras.layers.Dense(
            d_model
        )

        # Final projection
        self.output_dense = tf.keras.layers.Dense(
            d_model
        )

    def split_heads(
        self,
        x,
        batch_size
    ):

        # [batch, seq_len, d_model]
        x = tf.reshape(
            x,
            (
                batch_size,
                -1,
                self.num_heads,
                self.depth
            )
        )

        # [batch, heads, seq_len, depth]
        return tf.transpose(
            x,
            perm=[0, 2, 1, 3]
        )

    def call(
        self,
        query,
        key,
        value,
        mask=None
    ):

        batch_size = tf.shape(query)[0]

        # Linear projections
        q = self.wq(query)
        k = self.wk(key)
        v = self.wv(value)

        # Split heads
        q = self.split_heads(
            q,
            batch_size
        )

        k = self.split_heads(
            k,
            batch_size
        )

        v = self.split_heads(
            v,
            batch_size
        )

        # Attention
        attention_output, attention_weights = (
            scaled_dot_product_attention(
                q,
                k,
                v,
                mask
            )
        )

        # [batch, heads, seq_len, depth]
        # →
        # [batch, seq_len, heads, depth]
        attention_output = tf.transpose(
            attention_output,
            perm=[0, 2, 1, 3]
        )

        # Concatenate heads
        concat_attention = tf.reshape(
            attention_output,
            (
                batch_size,
                -1,
                self.d_model
            )
        )

        # Final projection
        output = self.output_dense(
            concat_attention
        )

        return output, attention_weights

In [108]:
#FFN -> Every Transformer layer contains a position-wise feed-forward network.
def feed_forward_network(
    d_model,
    d_ff
):

    return tf.keras.Sequential([

        tf.keras.layers.Dense(
            d_ff,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            d_model
        )
    ])

In [109]:
#encoder layer
class EncoderLayer(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout_rate
    ):
        super().__init__()

        self.self_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        self.ffn = feed_forward_network(
            d_model,
            d_ff
        )

        self.norm1 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm2 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.dropout1 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.dropout2 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

    def call(
        self,
        x,
        training=False,
        mask=None
    ):

        # Self-attention
        attention_output, attention_weights = (
            self.self_attention(
                x,
                x,
                x,
                mask
            )
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        # Residual + LayerNorm
        out1 = self.norm1(
            x + attention_output
        )

        # Feed-forward
        ffn_output = self.ffn(
            out1
        )

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        # Residual + LayerNorm
        out2 = self.norm2(
            out1 + ffn_output
        )

        return out2, attention_weights

In [110]:
#encoder stack  we need multiple encoder layers
class Encoder(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        d_ff,
        vocab_size,
        max_length,
        dropout_rate
    ):
        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = TokenEmbedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = positional_encoding(
            max_length,
            d_model
        )

        self.dropout = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.encoder_layers = [

            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout_rate
            )

            for _ in range(num_layers)
        ]

    def call(
        self,
        x,
        training=False,
        mask=None
    ):

        seq_length = tf.shape(x)[1]

        # Token embedding
        x = self.embedding(x)

        # Add positional encoding
        x += self.pos_encoding[
            :seq_length
        ]

        x = self.dropout(
            x,
            training=training
        )

        attention_maps = []

        # Pass through encoder layers
        for layer in self.encoder_layers:

            x, attention_weights = layer(
                x,
                training=training,
                mask=mask
            )

            attention_maps.append(
                attention_weights
            )

        return x, attention_maps

In [111]:
#padding mask *Padding tokens should not participate in attention
def create_padding_mask(sequence):

    mask = tf.cast(
        tf.equal(sequence, PAD_ID),
        tf.float32
    )

    # [batch, seq_len]
    # →
    # [batch, 1, 1, seq_len]

    return mask[:, tf.newaxis, tf.newaxis, :]

In [112]:
#test
sample = tf.constant([
    [10, 20, 30, 0, 0]
])

mask = create_padding_mask(sample)

print(mask.shape)
print(mask.numpy())

(1, 1, 1, 5)
[[[[0. 0. 0. 1. 1.]]]]


In [113]:
#look head mask
def create_look_ahead_mask(size):

    mask = 1 - tf.linalg.band_part(
        tf.ones(
            (size, size)
        ),
        -1,
        0
    )

    return mask

In [114]:
look_ahead = create_look_ahead_mask(5)

print(look_ahead.numpy())

[[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


In [115]:
#decoder layer
class DecoderLayer(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout_rate
    ):
        super().__init__()

        # Masked self-attention
        self.self_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        # Cross-attention
        self.cross_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        # Feed-forward
        self.ffn = feed_forward_network(
            d_model,
            d_ff
        )

        # Layer normalization
        self.norm1 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm2 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm3 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        # Dropout
        self.dropout1 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.dropout2 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.dropout3 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

    def call(
        self,
        x,
        encoder_output,
        training=False,
        look_ahead_mask=None,
        padding_mask=None
    ):

        # ==================================
        # 1. Masked Self Attention
        # ==================================

        self_attention_output, self_attention_weights = (
            self.self_attention(
                x,
                x,
                x,
                look_ahead_mask
            )
        )

        self_attention_output = self.dropout1(
            self_attention_output,
            training=training
        )

        out1 = self.norm1(
            x + self_attention_output
        )

        # ==================================
        # 2. Cross Attention
        # ==================================
        #
        # Query = Decoder
        # Key   = Encoder
        # Value = Encoder
        #

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                out1,
                encoder_output,
                encoder_output,
                padding_mask
            )
        )

        cross_attention_output = self.dropout2(
            cross_attention_output,
            training=training
        )

        out2 = self.norm2(
            out1 + cross_attention_output
        )

        # ==================================
        # 3. Feed Forward
        # ==================================

        ffn_output = self.ffn(
            out2
        )

        ffn_output = self.dropout3(
            ffn_output,
            training=training
        )

        out3 = self.norm3(
            out2 + ffn_output
        )

        return (
            out3,
            self_attention_weights,
            cross_attention_weights
        )

In [116]:
#decoder stack
class DecoderLayer(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        dropout_rate
    ):
        super().__init__()

        # Masked self-attention
        self.self_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        # Cross-attention
        self.cross_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        # Feed-forward
        self.ffn = feed_forward_network(
            d_model,
            d_ff
        )

        # Layer normalization
        self.norm1 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm2 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        self.norm3 = (
            tf.keras.layers.LayerNormalization(
                epsilon=1e-6
            )
        )

        # Dropout
        self.dropout1 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.dropout2 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.dropout3 = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

    def call(
        self,
        x,
        encoder_output,
        training=False,
        look_ahead_mask=None,
        padding_mask=None
    ):

        # ==================================
        # 1. Masked Self Attention
        # ==================================

        self_attention_output, self_attention_weights = (
            self.self_attention(
                x,
                x,
                x,
                look_ahead_mask
            )
        )

        self_attention_output = self.dropout1(
            self_attention_output,
            training=training
        )

        out1 = self.norm1(
            x + self_attention_output
        )

        # ==================================
        # 2. Cross Attention
        # ==================================
        #
        # Query = Decoder
        # Key   = Encoder
        # Value = Encoder
        #

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                out1,
                encoder_output,
                encoder_output,
                padding_mask
            )
        )

        cross_attention_output = self.dropout2(
            cross_attention_output,
            training=training
        )

        out2 = self.norm2(
            out1 + cross_attention_output
        )

        # ==================================
        # 3. Feed Forward
        # ==================================

        ffn_output = self.ffn(
            out2
        )

        ffn_output = self.dropout3(
            ffn_output,
            training=training
        )

        out3 = self.norm3(
            out2 + ffn_output
        )

        return (
            out3,
            self_attention_weights,
            cross_attention_weights
        )

In [117]:
class Decoder(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        num_layers,
        d_model,
        num_heads,
        d_ff,
        vocab_size,
        max_length,
        dropout_rate
    ):
        super().__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = TokenEmbedding(
            vocab_size,
            d_model
        )

        self.pos_encoding = positional_encoding(
            max_length,
            d_model
        )

        self.dropout = (
            tf.keras.layers.Dropout(
                dropout_rate
            )
        )

        self.decoder_layers = [

            DecoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout_rate
            )

            for _ in range(num_layers)
        ]

    def call(
        self,
        x,
        encoder_output,
        training=False,
        look_ahead_mask=None,
        padding_mask=None
    ):

        seq_length = tf.shape(x)[1]

        # Token embedding
        x = self.embedding(x)

        # Positional encoding
        x += self.pos_encoding[
            :seq_length
        ]

        x = self.dropout(
            x,
            training=training
        )

        self_attention_maps = []
        cross_attention_maps = []

        # Decoder layers
        for layer in self.decoder_layers:

            (
                x,
                self_attention,
                cross_attention
            ) = layer(
                x,
                encoder_output,
                training=training,
                look_ahead_mask=look_ahead_mask,
                padding_mask=padding_mask
            )

            self_attention_maps.append(
                self_attention
            )

            cross_attention_maps.append(
                cross_attention
            )

        return (
            x,
            self_attention_maps,
            cross_attention_maps
        )

In [118]:
class Transformer(
    tf.keras.Model
):

    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        d_model,
        num_heads,
        d_ff,
        input_vocab_size,
        target_vocab_size,
        input_max_length,
        target_max_length,
        dropout_rate
    ):
        super().__init__()

        self.encoder = Encoder(
            num_layers=num_encoder_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            vocab_size=input_vocab_size,
            max_length=input_max_length,
            dropout_rate=dropout_rate
        )

        self.decoder = Decoder(
            num_layers=num_decoder_layers,
            d_model=d_model,
            num_heads=num_heads,
            d_ff=d_ff,
            vocab_size=target_vocab_size,
            max_length=target_max_length,
            dropout_rate=dropout_rate
        )

        # Final vocabulary projection
        self.final_layer = tf.keras.layers.Dense(
            target_vocab_size
        )

    def call(
        self,
        encoder_inputs,
        decoder_inputs,
        training=False,
        encoder_padding_mask=None,
        decoder_look_ahead_mask=None,
        decoder_padding_mask=None
    ):

        # =========================
        # Encoder
        # =========================

        encoder_output, encoder_attention = (
            self.encoder(
                encoder_inputs,
                training=training,
                mask=encoder_padding_mask
            )
        )

        # =========================
        # Decoder
        # =========================

        decoder_output, decoder_self_attention, decoder_cross_attention = (
            self.decoder(
                decoder_inputs,
                encoder_output,
                training=training,
                look_ahead_mask=decoder_look_ahead_mask,
                padding_mask=decoder_padding_mask
            )
        )

        # =========================
        # Final Linear Projection
        # =========================

        logits = self.final_layer(
            decoder_output
        )

        return (
            logits,
            encoder_attention,
            decoder_self_attention,
            decoder_cross_attention
        )

In [119]:
#create transformer
transformer = Transformer(

    num_encoder_layers=NUM_ENCODER_LAYERS,

    num_decoder_layers=NUM_DECODER_LAYERS,

    d_model=D_MODEL,

    num_heads=NUM_HEADS,

    d_ff=D_FF,

    input_vocab_size=ENGLISH_VOCAB_SIZE,

    target_vocab_size=HINDI_VOCAB_SIZE,

    input_max_length=MAX_ENGLISH_LEN,

    target_max_length=MAX_HINDI_LEN,

    dropout_rate=DROPOUT_RATE
)

In [120]:
#create attention masks
#1. Encoder padding mask
#2. Decoder look-ahead + padding mask
#3. Decoder cross-attention padding mask

def create_masks(
    encoder_inputs,
    decoder_inputs
):

    # ==================================
    # Encoder padding mask
    # ==================================

    encoder_padding_mask = create_padding_mask(
        encoder_inputs
    )

    # ==================================
    # Decoder padding mask
    # ==================================

    decoder_padding_mask = create_padding_mask(
        encoder_inputs
    )

    # ==================================
    # Decoder self-attention mask
    # ==================================

    decoder_target_padding_mask = create_padding_mask(
        decoder_inputs
    )

    target_length = tf.shape(
        decoder_inputs
    )[1]

    look_ahead_mask = create_look_ahead_mask(
        target_length
    )

    # Combine:
    # Look-ahead mask + target padding mask

    combined_decoder_mask = tf.maximum(
        decoder_target_padding_mask,
        look_ahead_mask[
            tf.newaxis,
            tf.newaxis,
            :, :
        ]
    )

    return (
        encoder_padding_mask,
        combined_decoder_mask,
        decoder_padding_mask
    )

In [121]:
#load one real training chunk
chunk = np.load(
    chunk_files[0]
)

encoder_inputs = chunk[
    "encoder_inputs"
]

decoder_inputs = chunk[
    "decoder_inputs"
]

decoder_targets = chunk[
    "decoder_targets"
]

print("Encoder:", encoder_inputs.shape)
print("Decoder input:", decoder_inputs.shape)
print("Decoder target:", decoder_targets.shape)

Encoder: (100000, 56)
Decoder input: (100000, 64)
Decoder target: (100000, 64)


In [122]:
TEST_BATCH_SIZE = 4
encoder_batch = tf.convert_to_tensor(
    encoder_inputs[:TEST_BATCH_SIZE],
    dtype=tf.int32
)

decoder_batch = tf.convert_to_tensor(
    decoder_inputs[:TEST_BATCH_SIZE],
    dtype=tf.int32
)

target_batch = tf.convert_to_tensor(
    decoder_targets[:TEST_BATCH_SIZE],
    dtype=tf.int32
)

print("Encoder batch:", encoder_batch.shape)
print("Decoder batch:", decoder_batch.shape)
print("Target batch:", target_batch.shape)

Encoder batch: (4, 56)
Decoder batch: (4, 64)
Target batch: (4, 64)


In [123]:
(
    encoder_padding_mask,
    decoder_look_ahead_mask,
    decoder_padding_mask
) = create_masks(
    encoder_batch,
    decoder_batch
)

print(
    "Encoder mask:",
    encoder_padding_mask.shape
)

print(
    "Decoder look-ahead mask:",
    decoder_look_ahead_mask.shape
)

print(
    "Decoder padding mask:",
    decoder_padding_mask.shape
)

Encoder mask: (4, 1, 1, 56)
Decoder look-ahead mask: (4, 1, 64, 64)
Decoder padding mask: (4, 1, 1, 56)


In [124]:
(
    logits,
    encoder_attention,
    decoder_self_attention,
    decoder_cross_attention
) = transformer(
    encoder_batch,
    decoder_batch,
    training=False,
    encoder_padding_mask=encoder_padding_mask,
    decoder_look_ahead_mask=decoder_look_ahead_mask,
    decoder_padding_mask=decoder_padding_mask
)

print("Transformer built successfully.")
print("Logits shape:", logits.shape)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_12' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_13' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_6' (of type Encoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Transformer built successfully.
Logits shape: (4, 64, 16000)


In [125]:
print("Logits shape:", logits.shape)

Logits shape: (4, 64, 16000)


In [126]:
#build model
transformer.build(
    input_shape=[
        (None, MAX_ENGLISH_LEN),
        (None, MAX_HINDI_LEN)
    ]
)

transformer.summary()

Model: "transformer_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_6 (Encoder)             │ ?                      │     2,444,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_1 (Decoder)             │ ?                      │     2,577,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_125 (Dense)               │ (4, 64, 16000)         │     2,064,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,085,696 (27.03 MB)

 Trainable params: 7,085,696 (27.03 MB)

 Non-trainable params: 0 (0.00 B)

In [127]:
assert logits.shape[-1] == HINDI_VOCAB_SIZE

print(
    "Output vocabulary size:",
    logits.shape[-1]
)

print("Expected:", HINDI_VOCAB_SIZE)

Output vocabulary size: 16000
Expected: 16000


In [128]:
print(
    "First decoder token:",
    decoder_batch[0, 0].numpy()
)

print(
    "Expected BOS:",
    BOS_ID
)

assert (
    decoder_batch[:, 0] == BOS_ID
).numpy().all()

print("BOS check passed!")

First decoder token: 2
Expected BOS: 2
BOS check passed!


In [129]:
print(
    "First decoder token:",
    decoder_batch[0, 0].numpy()
)

print(
    "Expected BOS:",
    BOS_ID
)

assert (
    decoder_batch[:, 0] == BOS_ID
).numpy().all()

print("BOS check passed!")

First decoder token: 2
Expected BOS: 2
BOS check passed!


In [130]:
#Test a Random Forward Pass
random_encoder = tf.random.uniform(
    (
        2,
        MAX_ENGLISH_LEN
    ),
    minval=0,
    maxval=ENGLISH_VOCAB_SIZE,
    dtype=tf.int32
)

random_decoder = tf.random.uniform(
    (
        2,
        MAX_HINDI_LEN
    ),
    minval=0,
    maxval=HINDI_VOCAB_SIZE,
    dtype=tf.int32
)

In [131]:
(
    enc_mask,
    dec_mask,
    cross_mask
) = create_masks(
    random_encoder,
    random_decoder
)

In [132]:
random_logits, _, _, _ = transformer(
    random_encoder,
    random_decoder,
    training=False,
    encoder_padding_mask=enc_mask,
    decoder_look_ahead_mask=dec_mask,
    decoder_padding_mask=cross_mask
)

print(
    "Random test output:",
    random_logits.shape
)

Random test output: (2, 64, 16000)


In [133]:
#final architecture verification
expected_shape = (
    2,
    MAX_HINDI_LEN,
    HINDI_VOCAB_SIZE
)

assert (
    tuple(random_logits.shape)
    == expected_shape
)

print("================================")
print("TRANSFORMER TEST PASSED")
print("================================")

print("Encoder layers :", NUM_ENCODER_LAYERS)
print("Decoder layers :", NUM_DECODER_LAYERS)
print("Model dimension :", D_MODEL)
print("Attention heads :", NUM_HEADS)
print("FFN dimension   :", D_FF)
print("English vocab   :", ENGLISH_VOCAB_SIZE)
print("Hindi vocab     :", HINDI_VOCAB_SIZE)
print("English length  :", MAX_ENGLISH_LEN)
print("Hindi length    :", MAX_HINDI_LEN)
print("Parameters      :", f"{transformer.count_params():,}")

TRANSFORMER TEST PASSED
Encoder layers : 2
Decoder layers : 2
Model dimension : 128
Attention heads : 4
FFN dimension   : 512
English vocab   : 16000
Hindi vocab     : 16000
English length  : 56
Hindi length    : 64
Parameters      : 7,085,696


In [134]:
import os

BASE_PATH = "/content/drive/MyDrive/Translation Language Translator "
MODEL_PATH = os.path.join(BASE_PATH, "models")

os.makedirs(MODEL_PATH, exist_ok=True)

print("Model folder:", MODEL_PATH)
print("Exists:", os.path.exists(MODEL_PATH))

Model folder: /content/drive/MyDrive/Translation Language Translator /models
Exists: True


In [135]:
WEIGHTS_PATH = os.path.join(
    MODEL_PATH,
    "transformer_initial.weights.h5"
)

transformer.save_weights(WEIGHTS_PATH)

print("Weights saved to:")
print(WEIGHTS_PATH)

Weights saved to:
/content/drive/MyDrive/Translation Language Translator /models/transformer_initial.weights.h5


In [136]:
import json

config = {
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "num_encoder_layers": NUM_ENCODER_LAYERS,
    "num_decoder_layers": NUM_DECODER_LAYERS,
    "d_ff": D_FF,
    "dropout_rate": DROPOUT_RATE,
    "english_vocab_size": ENGLISH_VOCAB_SIZE,
    "hindi_vocab_size": HINDI_VOCAB_SIZE,
    "max_english_len": MAX_ENGLISH_LEN,
    "max_hindi_len": MAX_HINDI_LEN,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID
}

CONFIG_PATH = os.path.join(
    MODEL_PATH,
    "transformer_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved:")
print(CONFIG_PATH)

Configuration saved:
/content/drive/MyDrive/Translation Language Translator /models/transformer_config.json



                  ENGLISH
                     │
                     ▼
              Token Embedding
                     │
                     ▼
            Positional Encoding
                     │
                     ▼
        ┌────────────────────────┐
        │      ENCODER × 2       │
        │                        │
        │ Multi-Head Attention   │
        │          ↓             │
        │      Add + Norm        │
        │          ↓             │
        │ Feed Forward Network   │
        │          ↓             │
        │      Add + Norm        │
        └───────────┬────────────┘
                    │
                    ▼
             Encoder Output
                    │
                    │
                    ▼
             ┌───────────────┐
             │  DECODER × 2  │
             │               │
             │ Masked Self   │
             │ Attention     │
             │      ↓        │
             │ Add + Norm    │
             │      ↓        │
             │ Cross         │◄──────── Encoder Output
             │ Attention     │
             │      ↓        │
             │ Add + Norm    │
             │      ↓        │
             │ Feed Forward  │
             │      ↓        │
             │ Add + Norm    │
             └───────┬───────┘
                     │
                     ▼
              Linear Layer
                     │
                     ▼
             16,000 Hindi IDs
                     │
                     ▼
                Prediction

Notebook 3 implements a complete Transformer architecture from scratch using TensorFlow/Keras. The architecture consists of token embeddings, sinusoidal positional encoding, scaled dot-product attention, multi-head attention, feed-forward networks, residual connections, layer normalization, encoder and decoder stacks, padding masks, look-ahead masks, and encoder-decoder cross-attention. The model was tested using real tokenized English-Hindi data from the IIT Bombay Parallel Corpus and verified through forward-pass and tensor-shape checks. No pretrained Transformer model is used. The model is now ready for training in Notebook 4.